# **Computer Vision libraries in Pytorch**

In [ ]:
# Importing pytorch
import torch
from torch import nn

# Importing torchvision
import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor

# Importing matplotlib
import matplotlib.pyplot as plt

# Checking versions
print(torch.__version__)
print(torchvision.__version__)

In [ ]:
# Getting the training and testing data
train_data = datasets.FashionMNIST(root="data",
                                   train=True,
                                   download=True,
                                   transform=ToTensor(),
                                   target_transform=None
)

test_data = datasets.FashionMNIST(root="data",
                                  train=False,
                                  download=True,
                                  transform=ToTensor()
)

In [ ]:
image, label = train_data[0]
image, label

# **Input and Output shapes**

In [ ]:
image.shape, label

In [ ]:
len(train_data.data), len(train_data.targets), len(test_data.data), len(test_data.targets)

In [ ]:
# Classes
classes = train_data.classes
classes

In [ ]:
class_to_idx = train_data.class_to_idx
class_to_idx

In [ ]:
train_data.targets

# **Visualizing the data**

In [ ]:
# Visualizing the data
import matplotlib.pyplot as plt
image, label = train_data[0]
print(f"Shape: {image.shape}")
plt.imshow(image.squeeze())
plt.title(label)

In [ ]:
plt.imshow(image.squeeze(), cmap="gray")
plt.title(classes[label])

In [ ]:
# Plot random images
torch.manual_seed(42)
fig = plt.figure(figsize=(9, 9))
rows, cols = 4, 4
for i in range(1, rows*cols + 1):
  random_idx = torch.randint(0, len(train_data), size=[1]).item()
  img, label = train_data[random_idx]
  fig.add_subplot(rows, cols, i)
  plt.imshow(img.squeeze(), cmap="gray")
  plt.title(classes[label])
  plt.axis(False)

# **Using the dataloader**

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_dataloader = DataLoader(dataset=train_data,
                              batch_size=BATCH_SIZE,
                              shuffle=True)

test_dataloader = DataLoader(dataset=test_data,
                             batch_size=BATCH_SIZE,
                             shuffle=False)

print(f"Dataloaders: {train_dataloader}, {test_dataloader}")
print(f"Length of Train dataloader: {len(train_dataloader)}, Batch size: {BATCH_SIZE}")
print(f"Length of Test dataloader: {len(test_dataloader)}, Batch size: {BATCH_SIZE}")

In [ ]:
train_features_batch, train_labels_batch = next(iter(train_dataloader))
train_features_batch, train_features_batch.shape, train_labels_batch, train_labels_batch.shape

In [ ]:
torch.manual_seed(42)
random_idx = torch.randint(0, len(train_features_batch), size=[1]).item()
img, label = train_features_batch[random_idx], train_labels_batch[random_idx]
plt.imshow(img.squeeze(), cmap="gray")
plt.title(classes[label])
plt.axis("Off")
print(f"Image size: {img.shape}")
print(f"Label: {label}, Label size: {label.shape}")

# **Building the model**

In [ ]:
# Device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Create a flatten layer
flatten_layer = nn.Flatten()

x = train_features_batch[0]

output = flatten_layer(x)

print(f"X: {x}, Shape: {x.shape}")
print(f"Output: {output}, Shape: {output.shape}")

In [ ]:
# Building the model
class FashionMNISTModel(nn.Module):
  def __init__(self, input_shape, hidden_units, output_shape):
    super().__init__()

    self.layers_stack = nn.Sequential(
        nn.Flatten(), # neural networks like inputs in vector form
        nn.Linear(in_features=input_shape, out_features=hidden_units),
        nn.Linear(in_features=hidden_units, out_features=output_shape)
    )

  def forward(self, x):
    return self.layers_stack(x)

In [ ]:
# Instantiating the model
torch.manual_seed(42)

model_0 = FashionMNISTModel(input_shape=784,
                            hidden_units=10,
                            output_shape=len(classes)).to(device)
model_0, model_0.parameters(), next(model_0.parameters()).device

# **Loss function and optimizer**

In [ ]:
# Importing helper_functions.py
import requests
from pathlib import Path

if Path("helper_functions.py").is_file():
  print("helper_functions.py already exists")
else:
  print("Download helper_functions.py")
  request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
  with open("helper_functions.py", "wb") as f:
    f.write(request.content)

In [ ]:
# Loss function and optimizer
from helper_functions import accuracy_fn

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(params=model_0.parameters(),
                            lr=0.1)

In [ ]:
# Timing the training and testing
from timeit import default_timer as timer

def print_train_time(start: float, end: float, device: torch.device=None):
  total_time = end - start
  print(f"Train time on {device}, {total_time:.3f} seconds")
  return total_time

In [ ]:
# Testing the timing function
start_time = timer()
# Code here
end_time = timer()
print_train_time(start_time, end_time, device="cpu")

# **Training and testing**

In [ ]:
# Training and testing
from tqdm.auto import tqdm

torch.manual_seed(42)
torch.cuda.manual_seed(42)

train_time_start_on_cpu = timer()

epochs = 3

for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch}\n--------")

  train_loss = 0
  for batch, (X,y) in enumerate(train_dataloader):
    model_0.train()

    y_preds = model_0(X)

    loss = loss_fn(y_preds, y)

    train_loss += loss

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

  if batch%400 == 0:
    print(f"Looked at {batch*len(X)}/{len(train_dataloader.dataset)} samples")

  train_loss /= len(train_dataloader)

  # Testing the model
  test_loss, test_acc = 0, 0
  model_0.eval()
  with torch.inference_mode():
    for X,y in test_dataloader:
      test_preds = model_0(X)

      test_loss += loss_fn(test_preds, y)

      test_acc += accuracy_fn(y_true=y, y_pred=test_preds.argmax(dim=1))

    test_loss /= len(test_dataloader)

    test_acc /= len(test_dataloader)

  print(f"\nTrain loss: {train_loss:.5f}, Test loss: {test_loss:.5f}, Test accuracy: {test_acc:.2f}%\n")

train_time_end_on_cpu = timer()
train_time_model_0 = print_train_time(start=train_time_start_on_cpu, end=train_time_end_on_cpu, device=str(next(model_0.parameters()).device))

# **Making predictions on the trained model_0**

In [ ]:
# Making predictions on the trained model_0
torch.manual_seed(42)

def eval_model(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, accuracy_fn):
  loss, acc = 0, 0

  model.eval()
  with torch.inference_mode():
    for X, y in tqdm(data_loader):
      X, y = X.to(device), y.to(device)

      y_preds = model(X)

      loss += loss_fn(y_preds, y)
      acc += accuracy_fn(y_true=y, y_pred=y_preds.argmax(dim=1))

    loss /= len(data_loader)
    acc /= len(data_loader)

  return {"model_name": model.__class__.__name__,
          "model_loss": loss.item(),
          "model_acc": acc}

model_0_results = eval_model(model=model_0, data_loader=test_dataloader, loss_fn=loss_fn, accuracy_fn=accuracy_fn)
model_0_results

# **Improving the model**

In [ ]:
# Device agnostic code
import torch
from torch import nn

device  = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
# Building the model_1
class FashionMNISTModel1(nn.Module):
  def __init__(self, input_shape, hidden_units, output_shape):
    super().__init__()

    self.layers_stack = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=input_shape, out_features=hidden_units),
        nn.ReLU(),
        nn.Linear(in_features=hidden_units, out_features=output_shape),
        nn.ReLU()
    )

  def forward(self, x: torch.Tensor):
    return self.layers_stack(x)

In [ ]:
# Instantiating the model
torch.manual_seed(42)
model_1 = FashionMNISTModel1(input_shape=784, hidden_units=8, output_shape=len(classes)).to(device)
model_1, next(model_1.parameters()).device

# **Loss function and optimizer**

In [ ]:
# Loss function and optimizer
from helper_functions import accuracy_fn
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(params=model_1.parameters(),
                            lr=0.1)

# **Functionizing training and testing loops**

In [ ]:
# Functionizing training and testing loops
def train_step(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module, optimizer: torch.optim.Optimizer,
               accuracy_fn, device: torch.device=device):
  train_loss, train_acc = 0, 0
  model.to(device)
  for batch, (X, y) in enumerate(data_loader):
    X, y = X.to(device), y.to(device)

    y_preds = model(X)

    loss = loss_fn(y_preds, y)
    train_loss += loss
    train_acc += accuracy_fn(y_true=y, y_pred=y_preds.argmax(dim=1))

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

  train_loss /= len(data_loader)
  train_acc /= len(data_loader)
  print(f"Train loss: {train_loss:.5f}, Train accuracy: {train_acc:.2f}%")

def test_step(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module, accuracy_fn,
              device: torch.device=device):
  test_loss, test_acc = 0, 0
  model.to(device)
  model.eval()
  with torch.inference_mode():
    for X, y in data_loader:
      X, y = X.to(device), y.to(device)

      test_preds = model(X)
      test_loss += loss_fn(test_preds, y)
      test_acc += accuracy_fn(y_true=y, y_pred=test_preds.argmax(dim=1))

    test_loss /= len(data_loader)
    test_acc /= len(data_loader)
    print(f"Test loss: {test_loss:.5f}, Test accuracy: {test_acc:.2f}%")

# **Training and testing the model**

In [ ]:
# Training and testing the model
torch.manual_seed(42)

from timeit import default_timer as timer

train_time_start_on_cpu = timer()

epochs = 3

for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch}\n--------")
  train_step(model=model_1, data_loader=train_dataloader, loss_fn=loss_fn, optimizer=optimizer, accuracy_fn=accuracy_fn)

  test_step(model=model_1, data_loader=test_dataloader, loss_fn=loss_fn, accuracy_fn=accuracy_fn)

train_time_end_on_cpu = timer()
total_train_time_model_1 = print_train_time(start=train_time_start_on_cpu, end=train_time_end_on_cpu, device=device)

In [ ]:
model_0_results

In [ ]:
train_time_model_0

# **Evaluating the model**

In [ ]:
# Evaluating the model
torch.manual_seed(42)
model_1_results = eval_model(model=model_1, data_loader=test_dataloader, loss_fn=loss_fn, accuracy_fn=accuracy_fn)
model_1_results

# **Building a Convolutional Neural Network(CNN)**

In [ ]:
import torch
from torch import nn

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
# Building a CNN
class FashionMNISTModel2(nn.Module):
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
    super().__init__()

    self.block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape, out_channels=hidden_units,
                  kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units,
                  kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units,
                  kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units,
                  kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*7*7, out_features=output_shape)
    )

  def forward(self, x):
    x = self.block_1(x)
    # print(f"Output shape of block_1: {x.shape}")
    x = self.block_2(x)
    # print(f"Output shape of block_2: {x.shape}")
    x = self.classifier(x)
    return x

torch.manual_seed(42)
model_2 = FashionMNISTModel2(input_shape=1, hidden_units=10, output_shape=len(classes)).to(device)
model_2

# **Stepping through Conv2d()**

In [ ]:
# Stepping through Conv2d()
torch.manual_seed(42)

images = torch.rand(size=(32, 3, 64, 64))
test_image = images[0]
print(f"Image batch shape: {images.shape} -> [batch_size, color_channels, height, width]")
print(f"Single image shape: {test_image.shape} -> [color_channels, height, width]")
print(f"Single image pixel values:\n{test_image}")

In [ ]:
torch.manual_seed(42)

conv2d = nn.Conv2d(in_channels=3, out_channels=10, kernel_size=(5, 5), stride=2, padding=0)

conv2d(test_image)

In [ ]:
test_image.unsqueeze(dim=0).shape

In [ ]:
conv2d(test_image.unsqueeze(dim=0)).shape

In [ ]:
conv2d.state_dict()

In [ ]:
# Checking the shape of weight and bias of conv2d
print(f"conv2d weight shape: {conv2d.weight.shape} -> [out_channels=10, in_channels=3, kernel_size=5, kernel_size=5]")
print(f"conv2d bias shape: {conv2d.bias.shape} -> [out_channels=10]")

# **Trick to find the input and output shapes**

In [ ]:
rand_image_tensor = torch.randn(size=(1, 28, 28))
rand_image_tensor

In [ ]:
test_shape = model_2(rand_image_tensor.unsqueeze(dim=0).to(device))
test_shape.shape

# Seeing the Pooling layer

In [ ]:
# Seeing the pooling layer
print(f"Test image original shape: {test_image.shape}")
print(f"Test image unsqueeze shape: {test_image.unsqueeze(dim=0).shape}")

test_image_conv2d = conv2d(test_image.unsqueeze(dim=0))
print(f"Test image through conv2d shape: {test_image_conv2d.shape}")

maxpool = nn.MaxPool2d(kernel_size=2)

test_image_maxpool = maxpool(test_image.unsqueeze(dim=0))
print(f"Test image through maxpool shape: {test_image_maxpool.shape}")

test_image_conv2d_maxpool = maxpool(test_image_conv2d)
print(f"Test image through maxpool after conv2d: {test_image_conv2d_maxpool.shape}")

# **Loss function and optimizer**

In [ ]:
# Loss function and optimizer
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(params=model_2.parameters(),
                            lr=0.1)

# **Training and testing model_2**

In [ ]:
# Training and testing model_2
from timeit import default_timer as timer
torch.manual_seed(42)

epochs = 3

train_time_start_model_2 = timer()

for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch}\n-------")

  train_step(model=model_2, data_loader=train_dataloader, loss_fn=loss_fn, optimizer=optimizer, accuracy_fn=accuracy_fn, device=device)

  test_step(model=model_2, data_loader=test_dataloader, loss_fn=loss_fn, accuracy_fn=accuracy_fn, device=device)

train_time_end_model_2 = timer()
total_train_time = print_train_time(start=train_time_start_model_2, end=train_time_end_model_2, device=device)


In [ ]:
model_0_results

In [ ]:
# Evaluating the model
torch.manual_seed(42)
model_2_results = eval_model(model=model_2, data_loader=test_dataloader, loss_fn=loss_fn, accuracy_fn=accuracy_fn)
model_2_results

# **Compare model results and training time**

In [ ]:
# Compare model results and training time
import pandas as pd

compare_results = pd.DataFrame([model_0_results,
                                model_1_results,
                                model_2_results])
compare_results

In [ ]:
# Plotting Performance-Speed tradeoff
compare_results.set_index("model_name")["model_acc"].plot(kind="barh")
plt.xlabel("accuracy (%)")
plt.ylabel("model")

# **Making predictions with trained model_2**

In [ ]:
# Making predictions with trained model_2
def make_predictions(model: torch.nn.Module, data: list, device: torch.device=device):
  pred_probs = []
  model.eval()
  with torch.inference_mode():
    for sample in data:
      sample = torch.unsqueeze(sample, dim=0).to(device)

      pred_logits = model(sample)

      pred_prob = torch.softmax(pred_logits.squeeze(), dim=0)

      pred_probs.append(pred_prob.cpu())

  return torch.stack(pred_probs) # Converts to tensor and adds extra dimension

In [ ]:
import random
random.seed(42)
test_samples = []
test_labels = []
for sample, label in random.sample(list(test_data), k=9):
  test_samples.append(sample)
  test_labels.append(label)

print(f"Test sample image shape: {test_samples[0].shape}\nTest sample label: {test_labels[0]} ({classes[test_labels[0]]})")

In [ ]:
# Making predictions with trained model_2
pred_probs = make_predictions(model=model_2, data=test_samples)
pred_probs[:2]

In [ ]:
pred_classes = pred_probs.argmax(dim=1)
pred_classes

In [ ]:
test_labels, pred_classes

# **Visualizing the predictions**

In [ ]:
# Visualizing the predictions
plt.figure(figsize=(9, 9))
n_rows = 3
n_cols = 3
for i, sample in enumerate(test_samples):
  plt.subplot(n_rows, n_cols, i+1)

  plt.imshow(sample.squeeze(), cmap="gray")

  pred_label = classes[pred_classes[i]]

  truth_label = classes[test_labels[i]]

  title_text = f"Pred: {pred_label}, Truth: {truth_label}"

  if pred_label == truth_label:
    plt.title(title_text, fontsize=10, c="g")
  else:
    plt.title(title_text, fontsize=10, c="r")
  plt.axis(False)

# **Plotting the confusion matrix**

In [ ]:
# Making predictions on all test data
from tqdm.auto import tqdm

y_preds = []
model_2.eval()
with torch.inference_mode():
  for X, y in tqdm(test_dataloader, desc="Making predictions"):
    y_logits = model_2(X)

    y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)

    y_preds.append(y_pred.cpu())

y_pred_tensor = torch.cat(y_preds)
y_pred_tensor

In [ ]:
import mlxtend
print(mlxtend.__version__)

In [ ]:
!pip install torchmetrics # run only if required

In [ ]:
# Importing torchmetrics and mlxtend
import torchmetrics

In [ ]:
# Plotting the confusion matrix
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

confmat = ConfusionMatrix(num_classes=len(classes), task="multiclass")
confmat_tensor = confmat(preds=y_pred_tensor, target=test_data.targets)

fig, ax = plot_confusion_matrix(conf_mat=confmat_tensor.numpy(), class_names=classes, figsize=(10, 7))

# **Saving and Loading the model**

In [ ]:
# Saving the model
from pathlib import Path

MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Pytorch_03_Computer_Vision_model_CNN.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model_2.state_dict(), f=MODEL_SAVE_PATH)

In [ ]:
# Loading the model
loaded_model_2 = FashionMNISTModel2(input_shape=1, hidden_units=10, output_shape=len(classes))

loaded_model_2.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

loaded_model_2.to(device)

loaded_model_2.state_dict()

In [ ]:
# Evalutating the loaded model
torch.manual_seed(42)

loaded_model_2_results = eval_model(model=loaded_model_2, data_loader=test_dataloader, loss_fn=loss_fn, accuracy_fn=accuracy_fn)

loaded_model_2_results

In [ ]:
model_2_results

In [ ]:
# Checking if the results are close to each other
torch.isclose(torch.tensor(model_2_results["model_loss"]), torch.tensor(loaded_model_2_results["model_loss"]),
              atol=1e-08, rtol=0.0001)